# Ethereum Guard — Notebook 02: Benchmark Results, Ablation Study & Error Analysis (TV3-05)

This notebook loads the standardized experimental artifacts produced by `run_experiments.py`:
- `artifacts/results/benchmark_comparison.csv`
- `artifacts/results/ablation_study.csv`
- `artifacts/results/error_cases.csv`
- `artifacts/checkpoints/best_tgae.pt` (60-epoch training curve & calibrated branch weights)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = Path('..').resolve()
RESULTS_DIR = ROOT / 'artifacts' / 'results'
CKPT_DIR = ROOT / 'artifacts' / 'checkpoints'

bench_df = pd.read_csv(RESULTS_DIR / 'benchmark_comparison.csv')
ablation_df = pd.read_csv(RESULTS_DIR / 'ablation_study.csv')
error_df = pd.read_csv(RESULTS_DIR / 'error_cases.csv')

print('Benchmark rows:', len(bench_df))
print('Ablation rows :', len(ablation_df))
print('Error cases   :', len(error_df))
bench_df[['model', 'protocol', 'pr_auc', 'roc_auc', 'f1_binary', 'precision', 'recall', 'num_eval_nodes', 'inference_latency_ms_per_wallet']]

## 1. Benchmark Comparison: Cumulative vs. Strict Unseen Evaluation

In [ ]:
plot_df = bench_df.dropna(subset=['pr_auc']).copy()
plot_df['label'] = plot_df['model'] + '\n(' + plot_df['protocol'] + ')'

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(plot_df))
width = 0.35
ax.bar(x - width/2, plot_df['pr_auc'], width, label='PR-AUC', color='#38bdf8')
ax.bar(x + width/2, plot_df['f1_binary'].fillna(0), width, label='F1-Binary', color='#10b981')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['label'], rotation=12, ha='right', fontsize=9)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score')
ax.set_title('Ethereum Guard — PR-AUC & F1 Across Evaluation Protocols')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 2. Modular TG-AE Training Convergence Curve (`best_tgae.pt`)

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best_tgae.pt', map_location='cpu', weights_only=False)
st = ckpt.get('state', {})
hist = pd.DataFrame(st.get('training_history_', []))

print(f"Calibrated weights: alpha={st.get('alpha_'):.4f}, beta={st.get('beta_'):.4f}, gamma={st.get('gamma_'):.4f}")
print(f"Best Val PR-AUC: {st.get('best_val_pr_auc_'):.4f} | Val Best F1: {st.get('val_best_f1_'):.4f} | tau*: {st.get('threshold_'):.4f}")

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(hist['epoch'], hist['train_loss'], color='#f43f5e', linewidth=2, label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train Loss', color='#f43f5e')
ax2 = ax1.twinx()
ax2.plot(hist['epoch'], hist['val_pr_auc'], color='#10b981', linewidth=2, label='Val PR-AUC (Cumulative)')
ax2.set_ylabel('Val PR-AUC', color='#10b981')
ax2.set_ylim(0.5, 1.02)
plt.title('Modular TG-AE (Abl-G) 60-Epoch Convergence')
ax1.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Error Analysis (False Positives & False Negatives on Test Split)

In [ ]:
print(error_df['error_type'].value_counts())
print('\nTop 10 Highest-Margin False Positives (typically CEX hot wallets & DeFi routers):')
error_df[error_df['error_type'] == 'FALSE_POSITIVE'][
    ['address', 'anomaly_score', 'score_margin', 'total_eth_sent', 'tx_sent_count', 'tx_recv_count', 'burstiness_index', 'is_strictly_unseen_from_train']
].head(10)

## 4. 7-Configuration Ablation Matrix (Abl-A → Abl-G)

In [ ]:
ablation_df